In [9]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer

# Load the raw messy dataset
df = pd.read_csv('messy_customer_data.csv')

print("Original data shape:", df.shape)
df.head()

Original data shape: (10000, 12)


,CustomerID,Name,Gender,Age,City,Signup_Date,Last_purchase_date,purchase_amount,feedback_score,email,Phone_number,Country
0,NaN,Ankit,F,36.0,Mumbai,31/12/2024,13/08/2025,-999.0,2.0,NaN,0,Canada
1,C2,Ravi,Female,NaN,Kolkata,NaN,NaN,NaN,-1.0,user1mail.com,abc123,Canada
2,3,Ravi,female,66.0,Ahmedabad,NaN,20/06/2023,NaN,10.0,user2@mail.com,abc123,India
3,C4,Ankit,male,44.0,Kolkata,NaN,13/09/2025,NaN,NaN,NaN,9316267914,USA
4,5,Rahul,Male,200.0,Ahmedabad,09/07/2025,NaN,-999.0,NaN,user4mail.com,9234603292,USA


In [10]:
print("Shape:", df.shape)
print("Duplicates:", df.duplicated().sum())
print("Null Values:", df.isnull().sum())
print(df.info())
print("Description:", df.describe())

Shape: (10000, 12)
Duplicates: 0
Null Values: CustomerID            2379
Name                  1658
Gender                2455
Age                   2576
City                  1130
Signup_Date           2529
Last_purchase_date    3286
purchase_amount       5033
feedback_score        2462
email                 5066
Phone_number          2444
Country               1704
dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   CustomerID          7621 non-null   str    
 1   Name                8342 non-null   str    
 2   Gender              7545 non-null   str    
 3   Age                 7424 non-null   float64
 4   City                8870 non-null   str    
 5   Signup_Date         7471 non-null   str    
 6   Last_purchase_date  6714 non-null   str    
 7   purchase_amount     4967 non-null   float64
 8   feedback_score      7538 

In [11]:
# Remove duplicate rows
df = df.drop_duplicates()

In [12]:
# Clean and Standardize Text Columns (Gender, City, Country)
if 'Gender' in df.columns:
    df['Gender'] = df['Gender'].astype(str).str.strip().str.capitalize()
    df['Gender'] = df['Gender'].replace({'F': 'Female', 'M': 'Male', 'Nan': 'Unknown', 'None': 'Unknown'})

if 'City' in df.columns:
    df['City'] = df['City'].astype(str).str.strip().str.title()
    df['City'] = df['City'].replace({'Nan': 'Unknown', 'None': 'Unknown'})

if 'Country' in df.columns:
    df['Country'] = df['Country'].astype(str).str.strip().str.title()
    df['Country'] = df['Country'].replace({'Usa': 'USA', 'U.S.A.': 'USA', 'Nan': 'Unknown'})

In [13]:
# Handle invalid/out-of-range numeric values first (convert them to NaN)
if 'purchase_amount' in df.columns:
    df.loc[df['purchase_amount'] < 0, 'purchase_amount'] = np.nan

if 'feedback_score' in df.columns:
    df.loc[(df['feedback_score'] < 0) | (df['feedback_score'] > 10), 'feedback_score'] = np.nan

if 'Age' in df.columns:
    df.loc[(df['Age'] < 0) | (df['Age'] > 120), 'Age'] = np.nan

In [14]:
 # Apply KNN Imputer for missing numeric values (Age, purchase_amount, feedback_score)
numeric_cols = ['Age', 'purchase_amount', 'feedback_score']
# Filter columns that actually exist in the dataframe
existing_num_cols = [col for col in numeric_cols if col in df.columns]

if existing_num_cols:
    # Initialize KNN Imputer
    knn_imputer = KNNImputer(n_neighbors=5)
    df[existing_num_cols] = knn_imputer.fit_transform(df[existing_num_cols])

# Handle missing Customer IDs
if 'CustomerID' in df.columns:
    df['CustomerID'] = df['CustomerID'].fillna('UNKNOWN_ID')

# Fill remaining missing categorical/text values with 'Unknown'
categorical_cols = ['Name', 'Gender', 'City', 'Country', 'email', 'Phone_number']
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

In [15]:
#  Clean and validate Email & Phone Numbers
if 'email' in df.columns:
    # If email doesn't contain '@', mark as invalid/unknown
    df.loc[~df['email'].str.contains('@', na=False), 'email'] = 'unknown@domain.com'
    df['email'] = df['email'].fillna('unknown@domain.com')

if 'Phone_number' in df.columns:
    # Replace non-numeric or short phone entries with 'Unknown'
    df.loc[df['Phone_number'].astype(str).str.len() < 7, 'Phone_number'] = 'Unknown'
    df['Phone_number'] = df['Phone_number'].fillna('Unknown')

In [16]:
# Convert date columns into proper datetime format
date_cols = ['Signup_Date', 'Last_purchase_date']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

C:\Users\Hp\AppData\Local\Temp\ipykernel_10676\3337970238.py:5: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = pd.to_datetime(df[col], errors='coerce')
C:\Users\Hp\AppData\Local\Temp\ipykernel_10676\3337970238.py:5: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = pd.to_datetime(df[col], errors='coerce')


In [17]:
# Save the cleaned dataframe to a new CSV file
df.to_csv('cleaned_customer_data.csv', index=False)

print("Success! Data cleaned using KNN and saved as 'cleaned_customer_data.csv'.")
print("Final cleaned data shape:", df.shape)
print("Remaining Null Values:\n", df.isnull().sum())

Success! Data cleaned using KNN and saved as 'cleaned_customer_data.csv'.
Final cleaned data shape: (10000, 12)
Remaining Null Values:
 CustomerID               0
Name                     0
Gender                   0
Age                      0
City                     0
Signup_Date           5004
Last_purchase_date    6652
purchase_amount          0
feedback_score           0
email                    0
Phone_number             0
Country                  0
dtype: int64
